# 📓 실습 02. Neo4j 그래프 데이터베이스 적재 및 에뮬레이션

이 실습에서는 추출된 반도체 CMP 지식 트리플렛 데이터를 파이썬 드라이버로 Neo4j에 그래프 구조로 멱등하게 적재하고 조회하는 파이프라인을 실습합니다. 실제 DB 미가동 시에도 동작하도록 모의 클라이언트(Mock Client) 및 시각화 모듈이 기본 빌드되어 있습니다.

In [ ]:
import os
import json
import networkx as nx
import matplotlib.pyplot as plt

# 한글 깨짐 방지 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 1. 트리플렛 데이터 로드
triplets_path = os.path.join("..", "02_knowledge_triplet_extraction", "data", "extracted_triplets.json")
with open(triplets_path, "r", encoding="utf-8") as f:
    triplets = json.load(f)
print(f"불러온 지식 트리플렛 수: {len(triplets)} 개")

In [ ]:
# 2. Neo4j 연동 세션 및 모의 드라이버(Mock Client) 정의
# (실제 DB 연결이 차단되었을 때의 Fallback 처리)
class MockNeo4jClient:
    def __init__(self):
        self.nodes = set()
        self.edges = []
        
    def run_merge(self, sub, rel, obj):
        self.nodes.add(sub)
        self.nodes.add(obj)
        self.edges.append((sub, obj, {"type": rel}))
        
    def get_networkx_graph(self):
        G = nx.DiGraph()
        for node in self.nodes:
            G.add_node(node)
        for u, v, d in self.edges:
            G.add_edge(u, v, type=d["type"])
        return G

db_client = MockNeo4jClient()

# 데이터 적재 루프 가동 (Cypher MERGE 문 동작 시뮬레이션)
for t in triplets:
    db_client.run_merge(t["subject"], t["relation"], t["object"])
    
print(f"모의 적재 완료! 총 {len(db_client.nodes)} 개 노드, {len(db_client.edges)} 개 엣지 생성됨.")

In [ ]:
# 3. matplotlib을 연동한 2D 토폴로지 지식 맵 시각화
G = db_client.get_networkx_graph()
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, k=0.8)

nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='lightblue', edgecolors='black')
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15)

# 관계 종류 라벨링 표시
edge_labels = nx.get_edge_attributes(G, 'type')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, font_color='red')

plt.title("반도체 CMP 설비 지식 그래프 (Mock Neo4j Topology)", fontsize=15)
plt.axis('off')
plt.tight_layout()
plt.show()